# Эмбеддинги USER-bge-m3 для матчинга товаров

**Ядро: `xakaton (torch 2.5 + GPU)`**, не дефолтное `Python 3`.
В образе по умолчанию torch 2.4.0 + transformers 5.15, а transformers 5.x требует
torch >= 2.5 и молча отключает PyTorch — `AutoModel` там работать не будет.
Ядро `xakaton` смотрит в `~/xakaton/venv` (torch 2.5.1), где всё совместимо.

Рядом с блокнотом лежит `preprocess.py` — тот же файл, что используется в инференсе
(единый источник правды по чистке).

Задача: `matches.parquet` — 365 654 пары товаров, бинарный таргет (25.7% положительных).
`items_human.parquet` — ровно те 711 304 товара, что встречаются в парах.

План: чистка → эмбеддинги на GPU → косинус по паре как zero-shot бейзлайн → macro PR-AUC.

## 1. Окружение

In [ ]:
import sys, time, json
import numpy as np
import polars as pl
import torch

print('python  ', sys.version.split()[0])
print('torch   ', torch.__version__)
print('cuda    ', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu     ', torch.cuda.get_device_name(0))
    print('память  ', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# в ядре xakaton уже стоит всё нужное; при необходимости:
# !{sys.executable} -m pip install -q polars

## 2. Чистка

`preprocess.py` — тот же модуль, что пойдёт в инференс. Ключевое: функция чистая и
детерминированная, поэтому текст, на котором построен эмбеддинг, гарантированно
совпадёт с тем, что подаётся в модель в рантайме.

In [ ]:
import os
os.environ['HF_HOME'] = '/tmp/hf'      # кэш модели в /tmp: там 139 GB, в ~ всего 8 GB (tmpfs)

DATA = '/home/jovyan/xakaton/data'
OUT = '/tmp/matching'
os.makedirs(OUT, exist_ok=True)
ITEMS = f'{DATA}/items_human.parquet'
MATCHES = f'{DATA}/matches.parquet'

from preprocess import preprocess_frame

t = time.time()
items = preprocess_frame(ITEMS)
print(f'{items.height} товаров очищено за {time.time() - t:.0f} c')
items.head(3)

In [ ]:
# как выглядит текст, который увидит модель
for row in items.head(3).iter_rows(named=True):
    print(row['text'][:300])
    print('-' * 100)

## 3. Длина в токенах — выбираем max_length

In [ ]:
from transformers import AutoTokenizer, AutoModel

MODEL = 'deepvk/USER-bge-m3'
tokenizer = AutoTokenizer.from_pretrained(MODEL)

sample = items['text'].sample(20_000, seed=0).to_list()
lens = np.array([len(x) for x in tokenizer(sample, add_special_tokens=True)['input_ids']])
for q in (50, 90, 95, 99, 100):
    print(f'p{q:<3} {np.percentile(lens, q):6.0f} токенов')

In [ ]:
# Обрезка хвоста безопасна: build_item_text ставит различающие атрибуты
# (бренд, артикул, тип, цвет, размер) в начало строки.
MAX_LENGTH = 256
print(f'доля текстов, которые обрежутся при max_length={MAX_LENGTH}: {(lens > MAX_LENGTH).mean():.1%}')

## 4. Модель

BGE-M3 использует **CLS-пулинг** и L2-нормализацию для плотного представления.
Проверяем это по конфигу модели, а не на веру.

In [ ]:
from huggingface_hub import hf_hub_download

try:
    cfg = json.load(open(hf_hub_download(MODEL, '1_Pooling/config.json')))
    print(cfg)
    assert cfg.get('pooling_mode_cls_token'), 'ожидался CLS-пулинг — поправить encode()'
    print('\nCLS-пулинг подтверждён')
except Exception as e:
    print('не удалось проверить конфиг пулинга:', e)

In [ ]:
model = AutoModel.from_pretrained(MODEL, torch_dtype=torch.float16)
model = model.cuda().eval()
DIM = model.config.hidden_size
print('размерность эмбеддинга:', DIM)

## 5. Кодирование

Тексты сортируются по длине: в батч попадают тексты схожего размера, паддинга почти нет.
На разнородных данных это даёт кратное ускорение. Порядок на выходе — исходный.

In [ ]:
from tqdm.auto import tqdm

@torch.inference_mode()
def encode(texts, batch_size=256, max_length=MAX_LENGTH, quiet=False):
    order = np.argsort([len(t) for t in texts])
    out = np.empty((len(texts), DIM), dtype=np.float16)
    for i in tqdm(range(0, len(order), batch_size), disable=quiet):
        idx = order[i:i + batch_size]
        enc = tokenizer([texts[j] for j in idx], padding=True, truncation=True,
                        max_length=max_length, return_tensors='pt').to('cuda')
        h = model(**enc).last_hidden_state[:, 0]          # CLS
        h = torch.nn.functional.normalize(h, p=2, dim=1)  # L2
        out[idx] = h.float().cpu().numpy().astype(np.float16)
    return out

In [ ]:
# замер скорости перед полным прогоном
bench = items['text'].head(20_000).to_list()
torch.cuda.synchronize(); t = time.time()
_ = encode(bench, quiet=True)
torch.cuda.synchronize(); dt = time.time() - t

rate = len(bench) / dt
print(f'{rate:.0f} текстов/сек -> ~{items.height / rate / 60:.1f} мин на {items.height} товаров')

In [ ]:
texts = items['text'].to_list()
t = time.time()
emb = encode(texts)
print(f'{emb.shape} за {(time.time() - t) / 60:.1f} мин, {emb.nbytes / 1e9:.2f} GB')

np.save(f'{OUT}/emb_human.npy', emb)
items.write_parquet(f'{OUT}/items_human_clean.parquet')   # порядок строк = порядок в emb
print('сохранено')

## 6. Zero-shot бейзлайн: косинус по паре

Эмбеддинги нормированы, поэтому косинус — это просто скалярное произведение.

In [ ]:
matches = pl.read_parquet(MATCHES)
pos = {item_id: i for i, item_id in enumerate(items['id'].to_list())}

missing = sum(1 for i in matches['id1'] if i not in pos) + sum(1 for i in matches['id2'] if i not in pos)
assert missing == 0, f'{missing} товаров из пар отсутствуют в эмбеддингах'

i1 = np.fromiter((pos[i] for i in matches['id1']), dtype=np.int64, count=matches.height)
i2 = np.fromiter((pos[i] for i in matches['id2']), dtype=np.int64, count=matches.height)

emb32 = emb.astype(np.float32)
cos = np.einsum('ij,ij->i', emb32[i1], emb32[i2])
y = matches['target'].to_numpy()

print(f'пар: {len(y)}, положительных: {y.mean():.1%}')
print(f'косинус: положительные {cos[y == 1].mean():.3f} | отрицательные {cos[y == 0].mean():.3f}')

### Метрика

Целевая метрика — **macro PR-AUC**: average precision считается внутри каждой категории
и усредняется, чтобы мелкие категории весили столько же, сколько крупные.

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

category = dict(zip(items['id'].to_list(), items['category'].to_list()))
pair_cat = np.array([category[i] for i in matches['id1']])

def macro_pr_auc(y_true, score, groups, min_size=50):
    per = {}
    for g in np.unique(groups):
        m = groups == g
        if m.sum() >= min_size and 0 < y_true[m].sum() < m.sum():
            per[g] = average_precision_score(y_true[m], score[m])
    return float(np.mean(list(per.values()))), per

overall_ap = average_precision_score(y, cos)
macro_ap, per_cat = macro_pr_auc(y, cos, pair_cat)

print(f'ROC-AUC          {roc_auc_score(y, cos):.4f}')
print(f'PR-AUC (общий)   {overall_ap:.4f}   (baseline = доля положительных {y.mean():.4f})')
print(f'MACRO PR-AUC     {macro_ap:.4f}')

In [ ]:
print('по категориям:')
for cat, ap in sorted(per_cat.items(), key=lambda kv: -kv[1]):
    print(f'  {ap:.4f}  {cat}')

## 7. Item-level валидационный сплит

Для следующих шагов (обучение классификатора над эмбеддингами) нужен честный сплит:
товар не должен попадать одновременно в train и val, иначе модель подглядывает.
Сплит детерминированный — по хешу id, воспроизводится без сохранения файлов.

In [ ]:
import hashlib

def item_side(item_id, val_frac=0.2, seed=42):
    h = hashlib.blake2b(f'{seed}:{item_id}'.encode(), digest_size=8).digest()
    return 'val' if int.from_bytes(h, 'big') / 2**64 < val_frac else 'train'

side = {i: item_side(i) for i in items['id'].to_list()}
s1 = np.array([side[i] for i in matches['id1']])
s2 = np.array([side[i] for i in matches['id2']])

is_val = (s1 == 'val') & (s2 == 'val')
is_train = (s1 == 'train') & (s2 == 'train')
print(f'train {is_train.sum()}, val {is_val.sum()}, отброшено (смешанные) {(~is_train & ~is_val).sum()}')

val_macro, _ = macro_pr_auc(y[is_val], cos[is_val], pair_cat[is_val])
print(f'\nMACRO PR-AUC на val: {val_macro:.4f}')

## 8. Итог

`emb_human.npy` + `emb_human_ids.parquet` — эмбеддинги в порядке строк `items`.

Дальше: косинус — это одна фича. Следующий шаг — добить признаками, которые
эмбеддинг не видит (совпадение бренда/артикула, конфликты числовых атрибутов),
и обучить над ними классификатор на train-сплите.